In [11]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np
import pandas as pd

In [12]:
# Load environment variables from .env file
load_dotenv("../private_data/.env")

host = os.getenv("HOST")
db = os.getenv("DB")
port = os.getenv("PORT")
role = os.getenv("ROLE")
pw = os.getenv("PASSWORD")
engine = create_engine(f"postgresql+psycopg2://{role}:{pw}@{host}:{port}/{db}")

In [31]:
# Define your column lists
# cols_deals = ['updated_at', 'deleted_at', 'id', 'legacy_partner_id', 'title', 'description', 'creators_requirement', 'hash_tags', 'status', 'deal_value', 'go_live_at', 'live_until', 'deal_type',
#               'images', 'accepts_international', 'accepted_countries', 'social_requirement_type_id', 'product_name', 'schedule_type', 'schedule_model', 'legacy_id', 'tags', 'gender', 'featured_image', 'company_id', 'partner_id']
# cols_comp = ['applicants_applications_count', 'cancelled_applications_count', 'company_locations', 'completed_applications_count', 'content_types', 'deal_created_at', 'deal_deleted_at', 'deal_id', 'deal_tags', 'deal_updated_at',
#              'first_application_at', 'last_application_at', 'live_since', 'main_image', 'min_social_media_followers', 'pending_applications_count', 'planned_applications_count', 'rejected_applications_count', 'total_company_locations']

# 1. Load Deals
query_deals = f"SELECT * FROM public.deals;"
df_deals = pd.read_sql(query_deals, engine)

# 2. Load Deals Computed with built-in date parsing
# We add deal_id::text as deal_id_str directly in the SQL string
# query_comp = f"""
#     SELECT {', '.join(cols_comp)}, deal_id::text AS deal_id_str 
#     FROM public.deals_computed;
# """

query_comp = f"SELECT * FROM public.deals_computed;"

df_deals_comp = pd.read_sql(
    query_comp,
    engine,
    parse_dates=['first_application_at', 'last_application_at']
)

deals_comp_cols = ['applicants_applications_count', 'content_types', 'deal_id', 'main_image', 'min_social_media_followers', 'deal_tags', 'live_since', 'first_application_at', 'last_application_at', 'company_locations']
# Merge
df_deals = pd.merge(df_deals_comp[deals_comp_cols], df_deals, left_on = 'deal_id', right_on = 'id')

# df_deals = pd.merge(df_deals,
#                     df_deals_comp,
#                     left_on='id', right_on='deal_id', how='left')

del df_deals_comp
# Text takes up a lot of memory, can drop since it is not needed
df_deals.drop(columns=['id'], inplace=True)

# df_deals = df_deals.add_suffix('_deals')




## Cast columns to saveable datatypes

In [32]:

df_deals["first_application_at"] = pd.to_datetime(
    df_deals["first_application_at"], errors='coerce')
df_deals["last_application_at"] = pd.to_datetime(
    df_deals["last_application_at"], errors='coerce')



df_deals['deal_id'] = df_deals['deal_id'].astype(str)
# df_deals['id'] = df_deals['id'].astype(str)
df_deals['company_id'] = df_deals['company_id'].astype(str)
df_deals['partner_id'] = df_deals['partner_id'].astype(str)

import json

def safe_json_dump(x):
    # Handle NaNs or None
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    # Convert list/dict to valid JSON string
    return json.dumps(x)

# Apply JSON serialization
df_deals['company_locations'] = df_deals['company_locations'].apply(safe_json_dump)

In [28]:
df_deals

,applicants_applications_count,content_types,deal_id,main_image,min_social_media_followers,deal_tags,live_since,first_application_at,last_application_at,company_locations,...,social_requirement_type_id,product_name,schedule_type,schedule_model,legacy_id,tags,gender,featured_image,company_id,partner_id
0,7,"[{'id': 38, 'name': 'Music', 'slug': 'Music-No...",019689e3-09ed-00c8-ada6-052b1041b584,uploads/deals/019689e3-0a14-ffff-1130-d620d8c2...,2500,None,2023-09-21 07:40:56.211091,2023-09-24 21:36:08.063257,2023-10-05 17:01:53.664059,"[{""id"": ""01KERHSC8805FW4D6D5GG90JMF"", ""name"": ...",...,2.0,,physical_specific_days_all_day_schedule,"{'indefinitely': False, 'date_schedule_end_at'...",172.0,None,None,None,019bb11c-b0d3-015e-baaf-b9fa5b347794,01992995-1769-0065-49aa-6a67b7f7af7d
1,0,"[{'id': 23, 'name': 'Food', 'slug': 'Pizza'}]",019bbd3e-f64e-00c8-0e08-32e34caaaef2,uploads/deals/019bbd3e-75fd-ffff-9bca-d9682beb...,2500,None,2026-01-14 16:02:58.879237,NaT,NaT,"[{""id"": ""01KERHSGAG05FMYGV8JYE2PEAT"", ""name"": ...",...,2.0,https://bestel.burgernshake.nl/,physical_every_day_all_day_schedule,"{'indefinitely': True, 'date_schedule_end_at':...",NaN,None,unisex,uploads/deals/featured/019bbd3e-97e0-ffff-3597...,019bb11c-c121-015e-13c7-cd9c8b1c88e4,0199714c-8971-0065-43d5-fca6c121ee7b
2,56,"[{'id': 29, 'name': 'UGC', 'slug': 'Film-Strip...",019a118c-fcd5-00c8-6a5c-2b771b7f18a0,uploads/deals/019a118c-a436-ffff-eaa7-e9db6105...,2500,None,2025-10-23 18:25:27.486337,2025-10-23 18:46:51.044149,2025-12-17 17:00:51.616012,None,...,2.0,https://japchristmas.eu/collections/alle-kerst...,online_indefinitely_schedule,{},NaN,None,None,None,None,01992996-8f9a-0065-30a1-aa9519cd8e27
3,0,"[{'id': 21, 'name': 'Fashion', 'slug': 'Dress'}]",019689e5-f489-00c8-8c26-06b3ea0961a5,uploads/deals/019689e5-f54b-ffff-dbed-637e9d2a...,5000,None,2023-11-28 14:59:51.626780,NaT,NaT,None,...,3.0,,online_indefinitely_schedule,{},487.0,None,None,None,None,01992995-2f32-0065-d8da-ac763ad1f309
4,32,"[{'id': 23, 'name': 'Food', 'slug': 'Pizza'}, ...",019689e3-293c-00c8-c1df-735b603c5363,uploads/deals/019689e3-2aac-ffff-196b-7e667fbc...,10000,None,2024-05-22 12:00:33.784921,2024-05-22 12:08:26.158210,2024-06-28 09:56:33.026989,"[{""id"": ""01KERHVBR505FMVR4FSDTRR6AQ"", ""name"": ...",...,4.0,,physical_every_day_all_day_schedule,"{'indefinitely': True, 'date_schedule_end_at':...",1132.0,None,None,None,019bb11d-aee7-015e-7a46-e31c2aefb012,01992995-7e65-0065-edfb-434b238ddad4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7272,23,"[{'id': 23, 'name': 'Food', 'slug': 'Pizza'}, ...",019a8e95-765c-00c8-3c2d-c82d439d6798,uploads/deals/019a8e8d-91f5-ffff-08c8-be10fa47...,1500,None,2025-11-16 21:32:31.363091,2025-11-16 22:52:14.069908,2026-01-21 06:00:18.685408,"[{""id"": ""01KERJ12X905FNHCM2X7CBZMR3"", ""name"": ...",...,1.0,https://www.jimix.nl/,physical_specific_days_specific_time_schedule,{'end_time': {'RFC3339': '2025-12-10T22:55:00Z...,NaN,None,None,None,019bb120-8b88-015e-c255-5acb9ca19b34,01992996-00e9-0065-81ca-dbae66b7d097
7273,6,"[{'id': 35, 'name': 'Mom', 'slug': 'Baby'}]",019bc149-164d-00c8-3fd0-4ba01f6a2d70,uploads/deals/019bc148-d932-ffff-2c30-8f1668c6...,1500,None,2026-01-20 13:14:26.151491,2026-01-20 23:53:40.243580,2026-01-27 22:06:39.486637,None,...,1.0,https://happykidsboutique.eu/,online_indefinitely_schedule,{},NaN,None,unisex,uploads/deals/featured/019bc149-077a-ffff-5a8b...,019bb7c5-325c-015e-5c9a-3447bec38f70,019b2165-c3cb-0065-2ac7-05958d3ba823
7274,41,"[{'id': 20, 'name': 'Beauty', 'slug': 'Heart'}...",019adef8-be38-00c8-79d3-7661c386cf03,uploads/deals/019adef8-a53d-ffff-95a6-fb7e39a9...,5000,None,2025-12-02 12:10:35.116208,2025-12-02 13:29:52.214366,2026-01-14 13:22:40.964880,None,...,3.0,https://parfumselect.nl/prime-regenera-ii-supe...,online_indefinitely_schedule,{},NaN,None,None,None,None,019ad919-4b0a-0065-06f7-821b102edea9
7275,7,"[{'id': 41, 'name': 'Activities', 'slug': 'per...",019bbcee-7663-00c8-dc81

In [33]:
df_deals.to_parquet('../data/raw/BARTER_DEALS.parquet')